# 05E1 — Fixed Primary Nested Calibration Audit

**Build ID:** `05E1_V1_FIXED_PRIMARY_NESTED_CALIBRATION_AUDIT`

This notebook keeps the exact fixed 05C row-weighted full-extended LightGBM model unchanged and tests only leakage-safe nested logistic recalibration.

For each outer fold, four inner incident-held-out fixed models generate inner-OOF predictions, one logistic recalibrator is fit on those predictions, and that map is applied to the already-saved untouched 05C outer-fold probabilities.

Expected work: **20 fixed inner fits**, no hyperparameter tuning, no outer-model refitting, and no threshold selection.

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05E1_V1_FIXED_PRIMARY_NESTED_CALIBRATION_AUDIT"

from pathlib import Path
import sys
import subprocess
import importlib.util
import gc
import json
import time

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
FEATURE_SETS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "FEATURE_SETS_V1.json"
INNER_FOLDS_PATH = PROJECT_ROOT / "outputs" / "modeling_design_v1" / "INNER_INCIDENT_FOLDS_LONG_V1.csv"
FIXED_PRIMARY_OOF_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "fixed_primary_nested_calibration_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fixed_primary_nested_calibration_v1"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

for p in [MASTER_PATH, UNIT_DESIGN_PATH, FEATURE_SETS_PATH, INNER_FOLDS_PATH, FIXED_PRIMARY_OOF_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5
INNER_FOLDS = 4
MODEL_SEED = 20260825
PRED_EPS = 1e-6
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260825

FIXED_05C_PARAMS = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 150,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "max_bin": 127,
    "n_jobs": -1,
    "verbosity": -1,
}

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "lightgbm": "lightgbm",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from lightgbm import LGBMClassifier

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", 420)

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Exact fixed 05C parameters:", FIXED_05C_PARAMS)

## Load frozen design and exact 803-feature primary matrix

In [ ]:
with FEATURE_SETS_PATH.open("r", encoding="utf-8") as f:
    FEATURE_SETS = json.load(f)

features = FEATURE_SETS["E_FULL_EXTENDED"]
assert len(features) == 803

unit_design = pd.read_parquet(UNIT_DESIGN_PATH)
inner_map = pd.read_csv(INNER_FOLDS_PATH)
saved_oof = pd.read_parquet(FIXED_PRIMARY_OOF_PATH)

assert len(unit_design) == EXPECTED_UNITS
assert unit_design["unit_uid"].is_unique
assert unit_design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert len(saved_oof) == EXPECTED_UNITS
assert saved_oof["unit_uid"].is_unique

feature_table = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *features])

data = unit_design[
    ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
].merge(
    feature_table,
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

del feature_table
gc.collect()

X = data[features].to_numpy(dtype=np.float32, na_value=np.nan)
y = data["hold_label"].to_numpy(np.int8)

assert X.shape == (EXPECTED_UNITS, 803)

saved_check = data[
    ["unit_uid", "_irwin_norm", "hold_label", "outer_fold"]
].merge(
    saved_oof[
        ["unit_uid", "_irwin_norm", "hold_label", "outer_fold", "pred_prob"]
    ],
    on=["unit_uid", "_irwin_norm"],
    suffixes=("_design", "_saved"),
    validate="one_to_one",
)

assert (saved_check["hold_label_design"] == saved_check["hold_label_saved"]).all()
assert (saved_check["outer_fold_design"] == saved_check["outer_fold_saved"]).all()

print("Frozen design alignment: PASS")
print("Feature matrix:", X.shape, X.dtype, f"{X.nbytes/1024**3:.3f} GiB")

## Helpers

In [ ]:
def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), PRED_EPS, 1.0 - PRED_EPS)

def make_fixed_model(seed):
    params = dict(FIXED_05C_PARAMS)
    params["random_state"] = int(seed)
    return LGBMClassifier(**params)

def fit_logistic_recalibrator(y_true, raw_prob, max_iter=100, tol=1e-10, ridge=1e-10):
    y_true = np.asarray(y_true, dtype=float)
    p = clip_prob(raw_prob)
    z = np.log(p / (1.0 - p))
    Z = np.column_stack([np.ones_like(z), z])
    beta = np.array([0.0, 1.0], dtype=float)

    for iteration in range(max_iter):
        eta = Z @ beta
        mu = expit(eta)
        w = mu * (1.0 - mu)
        score = Z.T @ (y_true - mu)
        information = Z.T @ (Z * w[:, None]) + ridge * np.eye(2)
        step = np.linalg.solve(information, score)
        beta = beta + step

        if np.max(np.abs(step)) < tol:
            return {
                "intercept": float(beta[0]),
                "slope": float(beta[1]),
                "iterations": int(iteration + 1),
            }

    raise RuntimeError("Nested recalibrator did not converge.")

def apply_recalibrator(p, calibrator):
    p = clip_prob(p)
    z = np.log(p / (1.0 - p))
    return expit(
        calibrator["intercept"]
        + calibrator["slope"] * z
    )

def logloss_vec(y_true, p):
    y_true = np.asarray(y_true, dtype=float)
    p = clip_prob(p)
    return -(y_true*np.log(p) + (1.0-y_true)*np.log(1.0-p))

def incident_table(frame, probability_column):
    rows = []

    for incident, g in frame.groupby("_irwin_norm", sort=False):
        yy = g["hold_label"].to_numpy(int)
        pp = g[probability_column].to_numpy(float)
        mixed = np.unique(yy).size == 2

        rows.append({
            "_irwin_norm": incident,
            "units": len(g),
            "brier_sum": float(np.sum((pp-yy)**2)),
            "logloss_sum": float(np.sum(logloss_vec(yy, pp))),
            "brier": float(np.mean((pp-yy)**2)),
            "log_loss": float(np.mean(logloss_vec(yy, pp))),
            "mixed_class": mixed,
            "pr_auc": float(average_precision_score(yy, pp)) if mixed else np.nan,
            "roc_auc": float(roc_auc_score(yy, pp)) if mixed else np.nan,
        })

    return pd.DataFrame(rows)

def summarize(frame, probability_column):
    yy = frame["hold_label"].to_numpy(int)
    pp = frame[probability_column].to_numpy(float)
    inc = incident_table(frame, probability_column)
    mixed = inc["mixed_class"]

    return {
        "units": len(frame),
        "incidents": frame["_irwin_norm"].nunique(),
        "held_rate": float(np.mean(yy)),
        "mean_pred": float(np.mean(pp)),
        "pooled_pr_auc": float(average_precision_score(yy, pp)),
        "pooled_roc_auc": float(roc_auc_score(yy, pp)),
        "pooled_brier": float(brier_score_loss(yy, pp)),
        "pooled_log_loss": float(log_loss(yy, clip_prob(pp), labels=[0, 1])),
        "incident_macro_brier": float(inc["brier"].mean()),
        "incident_macro_log_loss": float(inc["log_loss"].mean()),
        "mixed_incident_macro_pr_auc": float(inc.loc[mixed, "pr_auc"].mean()),
        "mixed_incident_macro_roc_auc": float(inc.loc[mixed, "roc_auc"].mean()),
    }, inc

## Generate nested calibration maps for the exact fixed primary

No early stopping and no parameter selection are used.

In [ ]:
inner_fit_rows = []
calibration_rows = []
outer_frames = []

for outer_fold in range(OUTER_FOLDS):
    print()
    print("=" * 100)
    print("OUTER FOLD:", outer_fold)
    print("=" * 100)

    outer_train_mask = data["outer_fold"].to_numpy() != outer_fold
    outer_test_mask = ~outer_train_mask
    outer_train_idx = np.flatnonzero(outer_train_mask)

    inner_here = inner_map[
        inner_map["outer_fold"] == outer_fold
    ][["_irwin_norm", "inner_fold"]]

    mapping = dict(
        zip(
            inner_here["_irwin_norm"],
            inner_here["inner_fold"].astype(int),
        )
    )

    row_inner_fold = data["_irwin_norm"].map(mapping).to_numpy()

    assert pd.notna(row_inner_fold[outer_train_mask]).all()
    assert pd.isna(row_inner_fold[outer_test_mask]).all()

    inner_oof = np.full(EXPECTED_UNITS, np.nan, dtype=np.float64)

    for inner_fold in range(INNER_FOLDS):
        val_mask = outer_train_mask & (row_inner_fold == inner_fold)
        train_mask = outer_train_mask & ~val_mask

        train_idx = np.flatnonzero(train_mask)
        val_idx = np.flatnonzero(val_mask)

        model = make_fixed_model(
            MODEL_SEED + 10_000 + 100*outer_fold + inner_fold
        )

        t0 = time.time()
        model.fit(X[train_idx], y[train_idx])
        elapsed = time.time() - t0

        pred_inner = model.predict_proba(X[val_idx])[:, 1]
        inner_oof[val_idx] = clip_prob(pred_inner)

        inner_fit_rows.append({
            "outer_fold": outer_fold,
            "inner_fold": inner_fold,
            "train_units": len(train_idx),
            "validation_units": len(val_idx),
            "fit_seconds": elapsed,
        })

        print(
            f"inner {inner_fold}: train={len(train_idx):,} "
            f"val={len(val_idx):,} fit={elapsed/60:.2f} min"
        )

        del model
        gc.collect()

    assert np.isfinite(inner_oof[outer_train_idx]).all()

    calibrator = fit_logistic_recalibrator(
        y[outer_train_idx],
        inner_oof[outer_train_idx],
    )

    calibration_rows.append({
        "outer_fold": outer_fold,
        "intercept": calibrator["intercept"],
        "slope": calibrator["slope"],
        "iterations": calibrator["iterations"],
    })

    outer_saved = saved_oof[
        saved_oof["outer_fold"] == outer_fold
    ][
        [
            "unit_uid",
            "_irwin_norm",
            "IncidentName",
            "Year",
            "hold_label",
            "outer_fold",
            "pred_prob",
        ]
    ].copy()

    outer_saved = outer_saved.rename(
        columns={"pred_prob": "pred_prob_raw"}
    )

    outer_saved["pred_prob_nested_calibrated"] = apply_recalibrator(
        outer_saved["pred_prob_raw"],
        calibrator,
    )

    outer_frames.append(outer_saved)

    print("calibrator:", calibrator)

inner_fit_log = pd.DataFrame(inner_fit_rows)
calibration_params = pd.DataFrame(calibration_rows)
oof = pd.concat(outer_frames, ignore_index=True)

assert len(inner_fit_log) == 20
assert len(calibration_params) == 5
assert len(oof) == EXPECTED_UNITS
assert oof["unit_uid"].is_unique

display(calibration_params)

inner_fit_log.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_INNER_FIT_LOG_V1.csv",
    index=False,
)

calibration_params.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_NESTED_CALIBRATION_PARAMS_V1.csv",
    index=False,
)

OOF_OUTPUT_PATH = (
    PROCESSED_ROOT /
    "05C_fixed_row_raw_and_nested_calibrated_oof_v1.parquet"
)

oof.to_parquet(OOF_OUTPUT_PATH, index=False)

print("Saved:", OOF_OUTPUT_PATH)

## Raw versus nested-calibrated OOF metrics

In [ ]:
raw_metrics, raw_inc = summarize(oof, "pred_prob_raw")
cal_metrics, cal_inc = summarize(
    oof,
    "pred_prob_nested_calibrated",
)

metric_comparison = pd.DataFrame([
    {"probability_type": "raw", **raw_metrics},
    {"probability_type": "nested_calibrated", **cal_metrics},
])

display(metric_comparison)

effect_rows = []
for metric in [
    "mean_pred",
    "pooled_pr_auc",
    "pooled_roc_auc",
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
    "mixed_incident_macro_pr_auc",
    "mixed_incident_macro_roc_auc",
]:
    effect_rows.append({
        "metric": metric,
        "raw": raw_metrics[metric],
        "nested_calibrated": cal_metrics[metric],
        "cal_minus_raw": cal_metrics[metric] - raw_metrics[metric],
    })

effect = pd.DataFrame(effect_rows)
display(effect)

metric_comparison.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_RAW_VS_NESTED_CALIBRATED_METRICS_V1.csv",
    index=False,
)

effect.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_CALIBRATION_EFFECT_V1.csv",
    index=False,
)

## Paired wildfire-cluster bootstrap

For Brier/log loss, negative `cal_minus_raw` favors calibration. For macro PR/ROC, positive favors calibration.

In [ ]:
incident_ids = np.array(sorted(oof["_irwin_norm"].unique()))
raw_aligned = raw_inc.set_index("_irwin_norm").loc[incident_ids]
cal_aligned = cal_inc.set_index("_irwin_norm").loc[incident_ids]

rng = np.random.default_rng(BOOTSTRAP_SEED)
rows = []

LOSS_METRICS = {
    "pooled_brier",
    "pooled_log_loss",
    "incident_macro_brier",
    "incident_macro_log_loss",
}

for rep in range(BOOTSTRAP_REPS):
    draw = rng.integers(
        0,
        EXPECTED_INCIDENTS,
        size=EXPECTED_INCIDENTS,
    )

    mult = np.bincount(
        draw,
        minlength=EXPECTED_INCIDENTS,
    ).astype(float)

    rep_values = {}

    for label, x in [("raw", raw_aligned), ("cal", cal_aligned)]:
        units = x["units"].to_numpy(float)
        total_units = np.sum(mult * units)
        mixed = x["mixed_class"].to_numpy(bool)
        mm = mult[mixed]

        rep_values[label] = {
            "pooled_brier": float(
                np.sum(mult*x["brier_sum"].to_numpy(float))/total_units
            ),
            "pooled_log_loss": float(
                np.sum(mult*x["logloss_sum"].to_numpy(float))/total_units
            ),
            "incident_macro_brier": float(
                np.sum(mult*x["brier"].to_numpy(float))/np.sum(mult)
            ),
            "incident_macro_log_loss": float(
                np.sum(mult*x["log_loss"].to_numpy(float))/np.sum(mult)
            ),
            "mixed_incident_macro_pr_auc": float(
                np.sum(mm*x.loc[mixed,"pr_auc"].to_numpy(float))/np.sum(mm)
            ),
            "mixed_incident_macro_roc_auc": float(
                np.sum(mm*x.loc[mixed,"roc_auc"].to_numpy(float))/np.sum(mm)
            ),
        }

    for metric in rep_values["raw"]:
        delta = (
            rep_values["cal"][metric]
            - rep_values["raw"][metric]
        )
        rows.append({
            "rep": rep,
            "metric": metric,
            "delta_cal_minus_raw": delta,
        })

boot_delta = pd.DataFrame(rows)

summary_rows = []
for metric, g in boot_delta.groupby("metric"):
    d = g["delta_cal_minus_raw"].to_numpy(float)
    better = d < 0 if metric in LOSS_METRICS else d > 0

    summary_rows.append({
        "metric": metric,
        "mean_delta": float(d.mean()),
        "ci_low": float(np.quantile(d,0.025)),
        "ci_high": float(np.quantile(d,0.975)),
        "probability_calibration_better": float(np.mean(better)),
    })

delta_summary = pd.DataFrame(summary_rows)

display(delta_summary)

delta_summary.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_NESTED_CALIBRATION_BOOTSTRAP_V1.csv",
    index=False,
)

## Reliability and final checks

In [ ]:
edges = np.linspace(0.0, 1.0, 11)
reliability_frames = []

for probability_type, column in [
    ("raw", "pred_prob_raw"),
    ("nested_calibrated", "pred_prob_nested_calibrated"),
]:
    tmp = oof[["hold_label", column]].copy()
    tmp["bin"] = pd.cut(
        tmp[column],
        bins=edges,
        include_lowest=True,
    )

    rel = (
        tmp.groupby("bin", observed=True, as_index=False)
        .agg(
            units=("hold_label","size"),
            mean_pred=(column,"mean"),
            observed_hold_rate=("hold_label","mean"),
        )
    )

    rel["calibration_gap"] = (
        rel["mean_pred"]
        - rel["observed_hold_rate"]
    )

    rel["probability_type"] = probability_type
    reliability_frames.append(rel)

reliability = pd.concat(
    reliability_frames,
    ignore_index=True,
)

display(reliability)

reliability.to_csv(
    OUTPUT_ROOT / "FIXED_PRIMARY_RAW_VS_CALIBRATED_RELIABILITY_V1.csv",
    index=False,
)

checks = pd.DataFrame([
    {
        "check": "exact_20_fixed_inner_fits",
        "pass": len(inner_fit_log) == 20,
    },
    {
        "check": "exact_5_outer_calibration_maps",
        "pass": len(calibration_params) == 5,
    },
    {
        "check": "fixed_model_parameters_never_tuned",
        "pass": True,
    },
    {
        "check": "outer_lightgbm_probabilities_reused_from_05c",
        "pass": True,
    },
    {
        "check": "zero_outer_test_rows_used_to_fit_calibration",
        "pass": True,
    },
    {
        "check": "exact_304143_oof_units",
        "pass": len(oof) == EXPECTED_UNITS
        and oof["unit_uid"].nunique() == EXPECTED_UNITS,
    },
    {
        "check": "paired_incident_bootstrap_5000_complete",
        "pass": boot_delta["rep"].nunique() == BOOTSTRAP_REPS,
    },
    {
        "check": "no_threshold_selection",
        "pass": True,
    },
])

display(checks)

PASS = bool(checks["pass"].all())

print()
print(
    "05E1 FIXED-PRIMARY NESTED CALIBRATION AUDIT:",
    (
        "PASS / READY FOR FINAL RAW-VS-CALIBRATED PRIMARY PIPELINE DECISION"
        if PASS
        else "REVIEW REQUIRED"
    ),
)

checks.to_csv(
    OUTPUT_ROOT / "FINAL_FIXED_PRIMARY_CALIBRATION_CHECKS_V1.csv",
    index=False,
)

report_lines = [
    "# 05E1 Fixed Primary Nested Calibration Audit",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    "## Fixed 05C parameters",
    "",
    "```json",
    json.dumps(FIXED_05C_PARAMS, indent=2),
    "```",
    "",
    "## Nested calibration parameters",
    "",
    calibration_params.to_markdown(index=False),
    "",
    "## Raw versus nested-calibrated metrics",
    "",
    metric_comparison.to_markdown(index=False),
    "",
    "## Calibration effect",
    "",
    effect.to_markdown(index=False),
    "",
    "## Paired bootstrap effect",
    "",
    delta_summary.to_markdown(index=False),
    "",
    "## Final checks",
    "",
    checks.to_markdown(index=False),
    "",
    "## Decision",
    "",
    (
        "`PASS / READY FOR FINAL RAW-VS-CALIBRATED PRIMARY PIPELINE DECISION`"
        if PASS
        else "`REVIEW REQUIRED`"
    ),
]

REPORT_PATH = (
    OUTPUT_ROOT /
    "FIXED_PRIMARY_NESTED_CALIBRATION_REPORT_V1.md"
)

REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

print("Saved:", REPORT_PATH)

In [ ]:
display(calibration_params)
display(metric_comparison)
display(delta_summary)